# 🤖 Getting Started with strands-robots

**strands-robots** gives AI agents the ability to control real robots and physics simulations.
Think of it as the bridge between language models and physical actuators — the agent says
"pick up the red cube" and the library translates that into motor commands.

In this notebook you'll learn:
- How to install the library (and only the parts you need)
- Why imports are fast even though we depend on PyTorch and MuJoCo
- How the library finds robot model files on your machine

## Installation

The library is modular — install only what you need:

In [ ]:
# Base package: policies + tools (no GPU, no physics engine)
# !pip install strands-robots

# Add MuJoCo simulation (notebooks 5 & 6)
# !pip install "strands-robots[sim-mujoco]"

# Add GR00T ZMQ client (connects to NVIDIA inference containers)
# !pip install "strands-robots[groot-service]"

# Add LeRobot for direct HF model inference (needs GPU)
# !pip install "strands-robots[lerobot]"

# Everything at once
!pip install "strands-robots[all]"

## Fast Imports via Lazy Loading

A robotics library typically pulls in PyTorch, NumPy, MuJoCo, and camera drivers
on `import`. That's a 5-10 second delay just to check a config.

`strands-robots` solves this with lazy loading: `import strands_robots` takes milliseconds.
Heavy dependencies are loaded **only when you actually use them**.

In [ ]:
import time

start = time.time()
import strands_robots
elapsed = (time.time() - start) * 1000

print(f"Import took {elapsed:.0f}ms")
print()

# These three are available instantly — they have zero heavy dependencies:
print(f"Policy:        {strands_robots.Policy}")
print(f"MockPolicy:    {strands_robots.MockPolicy}")
print(f"create_policy: {strands_robots.create_policy}")

In [ ]:
# Everything the library exports:
print("Public API:", strands_robots.__all__)
print()

# Heavy symbols like Robot, Gr00tPolicy are listed but NOT imported yet.
# They only load when you access them:
#   strands_robots.Robot       → triggers: import lerobot, torch, numpy
#   strands_robots.Gr00tPolicy → triggers: import numpy

## Dependency Management with `require_optional()`

Every optional import in the codebase goes through one function.
When something's missing, you get a helpful error — not a cryptic traceback.

In [ ]:
from strands_robots.utils import require_optional

# NumPy is a base dependency — always available
np = require_optional("numpy")
print(f"✅ numpy {np.__version__}")

# ZMQ is optional (only needed for GR00T service mode)
# If you haven't installed it, you get a clear install command:
try:
    zmq = require_optional(
        "zmq",
        pip_install="pyzmq",
        extra="groot-service",
        purpose="GR00T service inference",
    )
    print(f"✅ zmq {zmq.__version__}")
except ImportError as e:
    print(f"Expected error for missing optional dep:\n{e}")

## Where Does Everything Live?

Robot model files (URDFs, MJCFs, meshes) go in a standard location.
Override with the `STRANDS_ASSETS_DIR` environment variable if needed.

In [ ]:
from strands_robots.utils import get_base_dir, get_assets_dir, resolve_asset_path

print(f"Base directory:   {get_base_dir()}")
print(f"Assets directory: {get_assets_dir()}")
print()

# resolve_asset_path is smart about different path types:
print("Path resolution examples:")
print(f"  None → default:  {resolve_asset_path(None, 'so100')}")
print(f"  Relative:        {resolve_asset_path('my_robot')}")
print(f"  Absolute:        {resolve_asset_path('/tmp/model.xml')}")
print(f"  Home expansion:  {resolve_asset_path('~/custom/arm')}")

## What's Next?

Now that you have the library installed, head to **Notebook 02** to learn about
the **Policy abstraction** — the core idea that lets you swap between AI models
without changing your robot code.